# Síntesis final y trazabilidad del proyecto

## Verificación de los resultados consolidados

**Objetivo.** Se reúnen las principales evidencias generadas durante las fases de auditoría, preparación de los datos, análisis exploratorio y modelado. Además, se comprueba que la configuración almacenada coincide con la utilizada en la evaluación final.

**Criterio.** Este notebook no entrena modelos ni modifica las decisiones metodológicas adoptadas. Su función es exclusivamente documental: cargar los resultados definitivos, verificar su coherencia y generar una síntesis reproducible para la memoria del TFM.

La configuración definitiva está compuesta por una ventana de observación de 60 días, 176 variables predictoras, una regresión logística con regularización L2 y un umbral de clasificación de `0,20`.

In [ ]:
from pathlib import Path
import json
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn


# ---------------------------------------------------------
# Localización robusta del proyecto
# ---------------------------------------------------------

RUTA_ACTUAL = Path.cwd()

if (RUTA_ACTUAL / "resultados").exists():
    RUTA_PROYECTO = RUTA_ACTUAL

elif (RUTA_ACTUAL.parent / "resultados").exists():
    RUTA_PROYECTO = RUTA_ACTUAL.parent

else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz "
        "del proyecto_churn."
    )

RUTA_RESULTADOS = (
    RUTA_PROYECTO
    / "resultados"
)

RUTA_EVIDENCIAS = (
    RUTA_PROYECTO
    / "evidencias"
)

RUTA_MODELOS = (
    RUTA_PROYECTO
    / "modelos"
)

print(
    "Raíz del proyecto:",
    RUTA_PROYECTO
)


# ---------------------------------------------------------
# Archivos críticos del proyecto
# ---------------------------------------------------------

archivos_criticos = {
    "resumen_final": (
        RUTA_RESULTADOS
        / "modelado_resumen_final_60d.json"
    ),
    "configuracion_final": (
        RUTA_RESULTADOS
        / "modelado_configuracion_final_60d.json"
    ),
    "metricas_test": (
        RUTA_RESULTADOS
        / "modelado_metricas_test_externo_60d.csv"
    ),
    "comparacion_temporal": (
        RUTA_RESULTADOS
        / "modelado_comparacion_temporal_oof.csv"
    ),
    "coeficientes": (
        RUTA_RESULTADOS
        / "modelado_coeficientes_logistic_l2_60d.csv"
    ),
    "estabilidad_coeficientes": (
        RUTA_RESULTADOS
        / "modelado_estabilidad_coeficientes_60d.csv"
    ),
    "escenarios_operativos": (
        RUTA_RESULTADOS
        / "modelado_escenarios_operativos_60d.csv"
    ),
    "inventario_evidencias": (
        RUTA_RESULTADOS
        / "inventario_evidencias_modelado.csv"
    ),
    "modelo_final": (
        RUTA_MODELOS
        / "modelo_final_logistic_l2_60d.joblib"
    )
}

figuras_finales = {
    "comparacion_temporal": (
        RUTA_EVIDENCIAS
        / "figura_comparacion_ventanas_modelos.png"
    ),
    "curva_roc": (
        RUTA_EVIDENCIAS
        / "figura_curva_roc_test_60d.png"
    ),
    "curva_precision_recall": (
        RUTA_EVIDENCIAS
        / "figura_curva_precision_recall_test_60d.png"
    ),
    "matriz_confusion": (
        RUTA_EVIDENCIAS
        / "figura_matriz_confusion_test_60d.png"
    ),
    "coeficientes": (
        RUTA_EVIDENCIAS
        / "figura_coeficientes_principales_60d.png"
    )
}


# ---------------------------------------------------------
# Comprobación de existencia
# ---------------------------------------------------------

comprobacion_archivos = pd.DataFrame([
    {
        "tipo": "archivo",
        "elemento": nombre,
        "ruta": str(
            ruta.relative_to(
                RUTA_PROYECTO
            )
        ),
        "existe": ruta.exists(),
        "tamano_mb": (
            ruta.stat().st_size
            / (1024 ** 2)
            if ruta.exists()
            else np.nan
        )
    }
    for nombre, ruta
    in archivos_criticos.items()
] + [
    {
        "tipo": "figura",
        "elemento": nombre,
        "ruta": str(
            ruta.relative_to(
                RUTA_PROYECTO
            )
        ),
        "existe": ruta.exists(),
        "tamano_mb": (
            ruta.stat().st_size
            / (1024 ** 2)
            if ruta.exists()
            else np.nan
        )
    }
    for nombre, ruta
    in figuras_finales.items()
])

display(
    comprobacion_archivos.style.format({
        "tamano_mb": "{:.3f}"
    })
)

if not comprobacion_archivos[
    "existe"
].all():
    raise FileNotFoundError(
        "Faltan uno o varios archivos críticos."
    )


# ---------------------------------------------------------
# Carga de los resultados definitivos
# ---------------------------------------------------------

with open(
    archivos_criticos["resumen_final"],
    "r",
    encoding="utf-8"
) as archivo:
    resumen_final = json.load(
        archivo
    )

with open(
    archivos_criticos[
        "configuracion_final"
    ],
    "r",
    encoding="utf-8"
) as archivo:
    configuracion_final = json.load(
        archivo
    )

metricas_test = pd.read_csv(
    archivos_criticos[
        "metricas_test"
    ]
)

comparacion_temporal = pd.read_csv(
    archivos_criticos[
        "comparacion_temporal"
    ]
)

coeficientes_modelo = pd.read_csv(
    archivos_criticos[
        "coeficientes"
    ]
)

estabilidad_coeficientes = pd.read_csv(
    archivos_criticos[
        "estabilidad_coeficientes"
    ]
)

escenarios_operativos = pd.read_csv(
    archivos_criticos[
        "escenarios_operativos"
    ]
)

inventario_evidencias = pd.read_csv(
    archivos_criticos[
        "inventario_evidencias"
    ]
)

paquete_modelo = joblib.load(
    archivos_criticos[
        "modelo_final"
    ]
)


# ---------------------------------------------------------
# Síntesis de la configuración final
# ---------------------------------------------------------

configuracion_seleccionada = (
    resumen_final[
        "selected_configuration"
    ]
)

resumen_configuracion = pd.DataFrame([
    {
        "elemento": "Ventana de observación",
        "valor": (
            f"{configuracion_seleccionada['window_days']} días"
        )
    },
    {
        "elemento": "Modelo",
        "valor": configuracion_seleccionada[
            "model"
        ]
    },
    {
        "elemento": "Umbral",
        "valor": (
            f"{configuracion_seleccionada['threshold']:.2f}"
        )
    },
    {
        "elemento": "Variables predictoras",
        "valor": resumen_final[
            "dataset"
        ][
            "predictor_variables"
        ]
    },
    {
        "elemento": "Clientes de entrenamiento",
        "valor": resumen_final[
            "dataset"
        ][
            "original_train_clients"
        ]
    },
    {
        "elemento": "Clientes de test externo",
        "valor": resumen_final[
            "dataset"
        ][
            "external_test_clients"
        ]
    }
])

print(
    "\nConfiguración final:"
)

display(
    resumen_configuracion
)


# ---------------------------------------------------------
# Síntesis de métricas del test externo
# ---------------------------------------------------------

metricas_finales = (
    resumen_final[
        "external_test_metrics"
    ]
)

resumen_metricas = pd.DataFrame([
    {
        "metrica": "Average precision",
        "valor": metricas_finales[
            "average_precision"
        ]
    },
    {
        "metrica": "AUC-ROC",
        "valor": metricas_finales[
            "auc_roc"
        ]
    },
    {
        "metrica": "Precision",
        "valor": metricas_finales[
            "precision"
        ]
    },
    {
        "metrica": "Recall",
        "valor": metricas_finales[
            "recall"
        ]
    },
    {
        "metrica": "F1",
        "valor": metricas_finales[
            "f1"
        ]
    },
    {
        "metrica": "F2",
        "valor": metricas_finales[
            "f2"
        ]
    },
    {
        "metrica": "Tasa de alertas",
        "valor": metricas_finales[
            "alert_rate"
        ]
    },
    {
        "metrica": "Brier score",
        "valor": metricas_finales[
            "brier"
        ]
    },
    {
        "metrica": "Log loss",
        "valor": metricas_finales[
            "log_loss"
        ]
    }
])

print(
    "\nMétricas definitivas del test externo:"
)

display(
    resumen_metricas.style.format({
        "valor": "{:.6f}"
    })
)


# ---------------------------------------------------------
# Comprobaciones cruzadas
# ---------------------------------------------------------

fila_metricas_test = (
    metricas_test.iloc[0]
)

validaciones_sintesis = {
    "estado_modelado_cerrado": (
        resumen_final["status"]
        == "modeling_closed"
    ),
    "ventana_final_60_dias": (
        configuracion_seleccionada[
            "window_days"
        ] == 60
    ),
    "modelo_final_logistic_l2": (
        configuracion_seleccionada[
            "model"
        ] == "logistic_l2"
    ),
    "umbral_final_020": np.isclose(
        configuracion_seleccionada[
            "threshold"
        ],
        0.20
    ),
    "modelo_guardado_60_dias": (
        paquete_modelo[
            "ventana_dias"
        ] == 60
    ),
    "modelo_guardado_umbral_020": (
        np.isclose(
            paquete_modelo["umbral"],
            0.20
        )
    ),
    "modelo_guardado_176_variables": (
        len(
            paquete_modelo[
                "variables"
            ]
        ) == 176
    ),
    "metricas_csv_coinciden": (
        np.isclose(
            fila_metricas_test[
                "average_precision"
            ],
            metricas_finales[
                "average_precision"
            ]
        )
        and np.isclose(
            fila_metricas_test[
                "f2"
            ],
            metricas_finales["f2"]
        )
    ),
    "nueve_configuraciones_temporales": (
        len(comparacion_temporal) == 9
    ),
    "176_coeficientes": (
        len(coeficientes_modelo) == 176
    ),
    "176_variables_estabilidad": (
        len(estabilidad_coeficientes)
        == 176
    ),
    "tres_escenarios_operativos": (
        len(escenarios_operativos) == 3
    ),
    "cinco_figuras_finales": (
        len(figuras_finales) == 5
    )
}

display(
    pd.DataFrame({
        "comprobacion": (
            validaciones_sintesis.keys()
        ),
        "resultado": (
            validaciones_sintesis.values()
        )
    })
)


# ---------------------------------------------------------
# Información del entorno de ejecución
# ---------------------------------------------------------

entorno_ejecucion = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "sistema": platform.platform()
}

print(
    "\nEntorno de ejecución:"
)

display(
    pd.DataFrame(
        entorno_ejecucion.items(),
        columns=[
            "componente",
            "version"
        ]
    )
)


# ---------------------------------------------------------
# Almacenamiento de la síntesis verificada
# ---------------------------------------------------------

sintesis_verificada = {
    "status": (
        "final_evidence_verified"
    ),
    "configuration": (
        configuracion_seleccionada
    ),
    "external_test_metrics": (
        metricas_finales
    ),
    "confusion_matrix": (
        resumen_final[
            "confusion_matrix"
        ]
    ),
    "calibration": (
        resumen_final[
            "calibration"
        ]
    ),
    "operational_scenarios": (
        resumen_final[
            "operational_scenarios"
        ]
    ),
    "interpretability": (
        resumen_final[
            "interpretability"
        ]
    ),
    "methodological_limits": (
        resumen_final[
            "methodological_limits"
        ]
    ),
    "environment": (
        entorno_ejecucion
    ),
    "all_checks_passed": bool(
        all(
            validaciones_sintesis.values()
        )
    )
}

RUTA_SINTESIS_VERIFICADA = (
    RUTA_RESULTADOS
    / "sintesis_final_verificada.json"
)

with open(
    RUTA_SINTESIS_VERIFICADA,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        sintesis_verificada,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Validación del bloque
# ---------------------------------------------------------

sintesis_final_valida = (
    all(
        validaciones_sintesis.values()
    )
    and RUTA_SINTESIS_VERIFICADA.exists()
)

print(
    "\nEstado del modelado:",
    resumen_final["status"]
)

print(
    "Configuración verificada:",
    (
        f"{configuracion_seleccionada['window_days']} días"
        f" + {configuracion_seleccionada['model']}"
        f" + umbral "
        f"{configuracion_seleccionada['threshold']:.2f}"
    )
)

print(
    "Archivos críticos comprobados:",
    len(comprobacion_archivos)
)

print(
    "¿La síntesis final se ha verificado "
    "correctamente?",
    sintesis_final_valida
)

## Registro de decisiones metodológicas

**Objetivo.** Se documentan las principales decisiones metodológicas del proyecto, los criterios aplicados, los datos utilizados y el momento en el que se adoptó cada decisión.

Este registro permite diferenciar entre:

- Elementos determinados por el dataset original.
- Decisiones definidas en el diseño previo del proyecto.
- Criterios operativos incorporados durante la reconstrucción.
- Análisis descriptivos realizados después de congelar la configuración definitiva.

Todas las decisiones que intervienen en la selección de la configuración final se fijaron antes de consultar el conjunto externo de prueba. Los análisis posteriores de interpretabilidad, calibración y utilidad operativa tuvieron una finalidad exclusivamente descriptiva y no modificaron el modelo ni su configuración.

In [ ]:
# ---------------------------------------------------------
# Registro estructurado de decisiones metodológicas
# ---------------------------------------------------------

registro_decisiones = pd.DataFrame([
    {
        "etapa": "Partición de los datos",
        "decision": (
            "Conservar la separación original entre "
            "train y test"
        ),
        "criterio": (
            "El test externo debía permanecer sin utilizar "
            "durante la selección del modelo."
        ),
        "origen": "Estructura proporcionada por el dataset",
        "datos_utilizados": (
            "52.004 clientes train y 13.001 clientes test"
        ),
        "momento": "Antes del modelado",
        "resultado": (
            "Test externo reservado hasta la configuración final"
        )
    },
    {
        "etapa": "Validación interna",
        "decision": (
            "Separar el 15 % del train como holdout interno"
        ),
        "criterio": (
            "Disponer de una comprobación interna adicional "
            "sin utilizar el test externo."
        ),
        "origen": "Decisión operativa de la reconstrucción",
        "datos_utilizados": (
            "44.203 clientes de desarrollo y "
            "7.801 de holdout interno"
        ),
        "momento": "Antes de entrenar los modelos",
        "resultado": (
            "División estratificada con semilla 314159"
        )
    },
    {
        "etapa": "Ingeniería de variables",
        "decision": (
            "Construir indicadores temporales para perfiles, "
            "categorías y actividad global"
        ),
        "criterio": (
            "Representar volumen, intensidad, tendencia, "
            "recencia y actividad reciente."
        ),
        "origen": "Diseño técnico del proyecto",
        "datos_utilizados": (
            "12 perfiles de actividad diaria"
        ),
        "momento": "Antes del modelado",
        "resultado": (
            "178 variables candidatas"
        )
    },
    {
        "etapa": "Selección técnica de variables",
        "decision": (
            "Eliminar dos variables constantes"
        ),
        "criterio": (
            "Una variable sin variabilidad no puede contribuir "
            "a discriminar entre clientes."
        ),
        "origen": "Comprobación técnica reproducible",
        "datos_utilizados": (
            "65.005 clientes"
        ),
        "momento": "Antes del modelado",
        "resultado": (
            "176 variables predictoras finales"
        )
    },
    {
        "etapa": "Ventanas temporales",
        "decision": (
            "Comparar 30, 60 y 90 días"
        ),
        "criterio": (
            "Comprobar si la señal aparece tempranamente o "
            "requiere observar casi toda la serie."
        ),
        "origen": "Diseño temporal previo del proyecto",
        "datos_utilizados": (
            "Solo clientes del conjunto de desarrollo"
        ),
        "momento": (
            "Antes de seleccionar la configuración final"
        ),
        "resultado": (
            "La ventana de 30 días no fue equivalente; "
            "60 y 90 días sí"
        )
    },
    {
        "etapa": "Modelos candidatos",
        "decision": (
            "Comparar modelos lineales, árboles y boosting"
        ),
        "criterio": (
            "Contrastar interpretabilidad lineal con capacidad "
            "para representar relaciones no lineales."
        ),
        "origen": "Diseño metodológico del modelado",
        "datos_utilizados": (
            "Validación cruzada estratificada de cinco pliegues"
        ),
        "momento": "Antes de seleccionar finalistas",
        "resultado": (
            "Finalistas: logistic_l2, logistic_balanced y xgboost"
        )
    },
    {
        "etapa": "Métrica principal",
        "decision": (
            "Utilizar average precision para comparar "
            "configuraciones"
        ),
        "criterio": (
            "Evaluar conjuntamente precisión y cobertura de "
            "la clase positiva en un problema desbalanceado."
        ),
        "origen": (
            "Protocolo operativo de esta reconstrucción"
        ),
        "datos_utilizados": (
            "Predicciones fuera de muestra de desarrollo"
        ),
        "momento": (
            "Congelado antes de comparar 30, 60 y 90 días"
        ),
        "resultado": (
            "Métrica principal del protocolo temporal"
        )
    },
    {
        "etapa": "Selección del umbral",
        "decision": (
            "Maximizar F2 sobre predicciones OOF"
        ),
        "criterio": (
            "Dar mayor peso al recall sin ignorar "
            "la precisión."
        ),
        "origen": (
            "Protocolo operativo de esta reconstrucción"
        ),
        "datos_utilizados": (
            "Exclusivamente conjunto de desarrollo"
        ),
        "momento": (
            "Antes de consultar el test externo"
        ),
        "resultado": (
            "Umbral definitivo de 0,20"
        )
    },
    {
        "etapa": "Regla de equivalencia",
        "decision": (
            "Preferir la ventana más corta y el modelo más "
            "sencillo entre resultados equivalentes"
        ),
        "criterio": (
            "Una ventana más larga o un modelo más complejo "
            "deben aportar una mejora material."
        ),
        "origen": (
            "Protocolo operativo congelado antes "
            "de la comparación"
        ),
        "datos_utilizados": (
            "Average precision, AUC-ROC, F2 y recall OOF"
        ),
        "momento": (
            "Antes de conocer la ventana seleccionada"
        ),
        "resultado": (
            "60 días y regresión logística L2"
        )
    },
    {
        "etapa": "Configuración definitiva",
        "decision": (
            "Seleccionar 60 días, logistic_l2 y umbral 0,20"
        ),
        "criterio": (
            "Rendimiento equivalente a 90 días con XGBoost, "
            "mayor anticipación y mejor interpretabilidad."
        ),
        "origen": (
            "Aplicación automática de la regla congelada"
        ),
        "datos_utilizados": (
            "44.203 clientes de desarrollo"
        ),
        "momento": (
            "Antes de abrir el test externo"
        ),
        "resultado": (
            "Configuración final congelada"
        )
    },
    {
        "etapa": "Evaluación externa",
        "decision": (
            "Evaluar una sola vez y prohibir reajustes posteriores"
        ),
        "criterio": (
            "Preservar la independencia del test como estimación "
            "final de generalización."
        ),
        "origen": "Protocolo de validación",
        "datos_utilizados": (
            "13.001 clientes del test externo"
        ),
        "momento": (
            "Después de congelar toda la configuración"
        ),
        "resultado": (
            "F2 0,9886 y recall 0,9933"
        )
    },
    {
        "etapa": "Interpretabilidad y calibración",
        "decision": (
            "Analizar coeficientes, estabilidad, calibración "
            "y concentración del riesgo"
        ),
        "criterio": (
            "Explicar el comportamiento del modelo y su posible "
            "utilidad operativa."
        ),
        "origen": "Análisis descriptivo posterior",
        "datos_utilizados": (
            "Modelo cerrado y resultados ya evaluados"
        ),
        "momento": (
            "Después de la evaluación externa"
        ),
        "resultado": (
            "No se modificaron modelo, variables ni umbral"
        )
    },
    {
        "etapa": "Limitación principal",
        "decision": (
            "No afirmar una anticipación temporal estrictamente "
            "demostrada"
        ),
        "criterio": (
            "La construcción temporal original de la etiqueta "
            "churn no está completamente documentada."
        ),
        "origen": "Limitación del dataset",
        "datos_utilizados": (
            "Documentación y estructura disponibles"
        ),
        "momento": "Durante toda la interpretación",
        "resultado": (
            "Las conclusiones se limitan al test proporcionado"
        )
    }
])


# ---------------------------------------------------------
# Tabla resumida de los resultados definitivos
# ---------------------------------------------------------

matriz_confusion_resumen = (
    resumen_final[
        "confusion_matrix"
    ]
)

tabla_resultados_definitivos = pd.DataFrame([
    {
        "bloque": "Configuración",
        "indicador": "Ventana",
        "valor": "60 días"
    },
    {
        "bloque": "Configuración",
        "indicador": "Modelo",
        "valor": "Regresión logística L2"
    },
    {
        "bloque": "Configuración",
        "indicador": "Umbral",
        "valor": "0,20"
    },
    {
        "bloque": "Configuración",
        "indicador": "Variables",
        "valor": "176"
    },
    {
        "bloque": "Test externo",
        "indicador": "Average precision",
        "valor": (
            f"{metricas_finales['average_precision']:.4f}"
        )
    },
    {
        "bloque": "Test externo",
        "indicador": "AUC-ROC",
        "valor": (
            f"{metricas_finales['auc_roc']:.4f}"
        )
    },
    {
        "bloque": "Test externo",
        "indicador": "Precision",
        "valor": (
            f"{metricas_finales['precision']:.4f}"
        )
    },
    {
        "bloque": "Test externo",
        "indicador": "Recall",
        "valor": (
            f"{metricas_finales['recall']:.4f}"
        )
    },
    {
        "bloque": "Test externo",
        "indicador": "F1",
        "valor": (
            f"{metricas_finales['f1']:.4f}"
        )
    },
    {
        "bloque": "Test externo",
        "indicador": "F2",
        "valor": (
            f"{metricas_finales['f2']:.4f}"
        )
    },
    {
        "bloque": "Matriz de confusión",
        "indicador": "Verdaderos negativos",
        "valor": str(
            matriz_confusion_resumen[
                "true_negatives"
            ]
        )
    },
    {
        "bloque": "Matriz de confusión",
        "indicador": "Falsos positivos",
        "valor": str(
            matriz_confusion_resumen[
                "false_positives"
            ]
        )
    },
    {
        "bloque": "Matriz de confusión",
        "indicador": "Falsos negativos",
        "valor": str(
            matriz_confusion_resumen[
                "false_negatives"
            ]
        )
    },
    {
        "bloque": "Matriz de confusión",
        "indicador": "Verdaderos positivos",
        "valor": str(
            matriz_confusion_resumen[
                "true_positives"
            ]
        )
    }
])


# ---------------------------------------------------------
# Presentación
# ---------------------------------------------------------

print(
    "Decisiones metodológicas registradas:",
    len(registro_decisiones)
)

display(
    registro_decisiones[
        [
            "etapa",
            "decision",
            "origen",
            "momento",
            "resultado"
        ]
    ]
)

print(
    "\nResumen de resultados definitivos:"
)

display(
    tabla_resultados_definitivos
)


# ---------------------------------------------------------
# Almacenamiento de las tablas
# ---------------------------------------------------------

RUTA_REGISTRO_DECISIONES = (
    RUTA_RESULTADOS
    / "tabla_registro_decisiones_metodologicas.csv"
)

RUTA_TABLA_RESULTADOS_DEFINITIVOS = (
    RUTA_RESULTADOS
    / "tabla_resultados_definitivos_tfm.csv"
)

RUTA_SINTESIS_METODOLOGICA = (
    RUTA_RESULTADOS
    / "sintesis_metodologica_para_memoria.json"
)

RUTA_RESUMEN_MEMORIA_MD = (
    RUTA_RESULTADOS
    / "resumen_para_memoria.md"
)

registro_decisiones.to_csv(
    RUTA_REGISTRO_DECISIONES,
    index=False
)

tabla_resultados_definitivos.to_csv(
    RUTA_TABLA_RESULTADOS_DEFINITIVOS,
    index=False
)


# ---------------------------------------------------------
# Síntesis metodológica en JSON
# ---------------------------------------------------------

sintesis_metodologica = {
    "status": "methodological_traceability_completed",
    "registered_decisions": int(
        len(registro_decisiones)
    ),
    "final_configuration": {
        "window_days": 60,
        "model": "logistic_l2",
        "threshold": 0.20,
        "features": 176
    },
    "external_test_metrics": (
        metricas_finales
    ),
    "configuration_changed_after_test": False,
    "main_limitation": (
        "The temporal construction of the original churn "
        "label is not fully documented."
    ),
    "decisions": (
        registro_decisiones.to_dict(
            orient="records"
        )
    )
}

with open(
    RUTA_SINTESIS_METODOLOGICA,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        sintesis_metodologica,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# Resumen en Markdown para reutilizar en la memoria
# ---------------------------------------------------------

texto_resumen_memoria = f"""# Síntesis del modelado

## Configuración final

- Ventana de observación: 60 días.
- Variables predictoras: 176.
- Modelo: regresión logística con regularización L2.
- Umbral de clasificación: 0,20.
- Entrenamiento final: 52.004 clientes.
- Test externo: 13.001 clientes.

## Resultados del test externo

- Average precision: {metricas_finales['average_precision']:.4f}.
- AUC-ROC: {metricas_finales['auc_roc']:.4f}.
- Precision: {metricas_finales['precision']:.4f}.
- Recall: {metricas_finales['recall']:.4f}.
- F1: {metricas_finales['f1']:.4f}.
- F2: {metricas_finales['f2']:.4f}.
- Tasa de alertas: {metricas_finales['alert_rate']:.2%}.

La matriz de confusión contiene \
{matriz_confusion_resumen['true_negatives']} verdaderos negativos, \
{matriz_confusion_resumen['false_positives']} falsos positivos, \
{matriz_confusion_resumen['false_negatives']} falsos negativos y \
{matriz_confusion_resumen['true_positives']} verdaderos positivos.

## Criterio de selección

La configuración se seleccionó aplicando un protocolo definido antes de
consultar el test externo. Entre las configuraciones consideradas
equivalentes se prefirió la ventana más corta y, dentro de ella, el modelo
más sencillo e interpretable.

## Limitación principal

La etiqueta churn fue proporcionada por el dataset y su construcción
temporal original no está completamente documentada. Por ello, los
resultados demuestran capacidad predictiva sobre la partición externa
proporcionada, pero no prueban por sí solos una anticipación temporal
estricta en un entorno productivo.
"""

with open(
    RUTA_RESUMEN_MEMORIA_MD,
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(
        texto_resumen_memoria
    )


# ---------------------------------------------------------
# Validación final
# ---------------------------------------------------------

registro_metodologico_valido = (
    sintesis_final_valida
    and len(registro_decisiones) == 13
    and registro_decisiones[
        "etapa"
    ].notna().all()
    and registro_decisiones[
        "criterio"
    ].notna().all()
    and tabla_resultados_definitivos.shape[0]
        == 14
    and RUTA_REGISTRO_DECISIONES.exists()
    and RUTA_TABLA_RESULTADOS_DEFINITIVOS.exists()
    and RUTA_SINTESIS_METODOLOGICA.exists()
    and RUTA_RESUMEN_MEMORIA_MD.exists()
)

print(
    "\nDecisiones metodológicas registradas:",
    len(registro_decisiones)
)

print(
    "Configuración documentada:",
    "60 días + logistic_l2 + umbral 0.20"
)

print(
    "¿Se modificó el modelo después del test?",
    sintesis_metodologica[
        "configuration_changed_after_test"
    ]
)

print(
    "¿El registro metodológico se ha generado "
    "correctamente?",
    registro_metodologico_valido
)

## Mapa de evidencias y cierre técnico

**Objetivo.** Se relacionan las principales afirmaciones del proyecto con los archivos que permiten verificarlas y se deja constancia del cierre del proceso analítico.

El mapa de trazabilidad identifica, para cada resultado, el notebook en el que se genera, el archivo que lo documenta y su función dentro de la memoria del TFM. Además, se calcula la huella digital SHA-256 de los artefactos finales más relevantes para comprobar posteriormente su integridad.

Este bloque no realiza nuevos análisis ni modifica el modelo o su configuración. Su finalidad es exclusivamente documental y contribuye a garantizar la trazabilidad y reproducibilidad del proyecto.

In [ ]:
import hashlib
from datetime import datetime


# ---------------------------------------------------------
# Comprobación de los notebooks del proyecto
# ---------------------------------------------------------

RUTA_NOTEBOOKS = (
    RUTA_PROYECTO
    / "notebooks"
)

notebooks_esperados = [
    "01_auditoria_dataset.ipynb",
    "02_preparacion_dataset.ipynb",
    "03_analisis_exploratorio.ipynb",
    "04_modelado.ipynb",
    "05_sintesis_final.ipynb"
]

comprobacion_notebooks = pd.DataFrame([
    {
        "notebook": nombre,
        "ruta": str(
            (
                RUTA_NOTEBOOKS
                / nombre
            ).relative_to(
                RUTA_PROYECTO
            )
        ),
        "existe": (
            RUTA_NOTEBOOKS
            / nombre
        ).exists()
    }
    for nombre in notebooks_esperados
])

display(
    comprobacion_notebooks
)

if not comprobacion_notebooks[
    "existe"
].all():
    raise FileNotFoundError(
        "Falta uno o varios notebooks del proyecto."
    )


# ---------------------------------------------------------
# Mapa entre conclusiones y evidencias
# ---------------------------------------------------------

mapa_evidencias = pd.DataFrame([
    {
        "bloque": "Auditoría del dataset",
        "afirmacion_verificable": (
            "El dataset contiene 65.005 clientes, "
            "12 perfiles y 90 días de actividad."
        ),
        "notebook_origen": (
            "01_auditoria_dataset.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/clientes_base.csv"
        ),
        "uso_en_memoria": (
            "Descripción y calidad del dataset"
        )
    },
    {
        "bloque": "Ingeniería de variables",
        "afirmacion_verificable": (
            "Se generaron 178 variables candidatas "
            "y se eliminaron dos constantes."
        ),
        "notebook_origen": (
            "02_preparacion_dataset.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "resumen_seleccion_variables_178_90d.json"
        ),
        "uso_en_memoria": (
            "Justificación de las 176 variables"
        )
    },
    {
        "bloque": "Análisis exploratorio",
        "afirmacion_verificable": (
            "La prevalencia de churn es cercana al 23 % "
            "y se analizaron distribuciones y correlaciones."
        ),
        "notebook_origen": (
            "03_analisis_exploratorio.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/eda_resumen_general.json"
        ),
        "uso_en_memoria": (
            "Descripción exploratoria"
        )
    },
    {
        "bloque": "Validación cruzada",
        "afirmacion_verificable": (
            "Se compararon los modelos candidatos mediante "
            "cinco pliegues estratificados."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/modelado_resumen_cv_modelos.csv"
        ),
        "uso_en_memoria": (
            "Comparación inicial de algoritmos"
        )
    },
    {
        "bloque": "Comparación temporal",
        "afirmacion_verificable": (
            "Se compararon nueve combinaciones de "
            "ventana y modelo."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_comparacion_temporal_oof.csv"
        ),
        "uso_en_memoria": (
            "Selección de la ventana de observación"
        )
    },
    {
        "bloque": "Configuración final",
        "afirmacion_verificable": (
            "La configuración seleccionada utiliza "
            "60 días, logistic_l2 y umbral 0,20."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_configuracion_final_60d.json"
        ),
        "uso_en_memoria": (
            "Justificación del modelo definitivo"
        )
    },
    {
        "bloque": "Test externo",
        "afirmacion_verificable": (
            "El test externo contiene 13.001 clientes "
            "y se evaluó una sola vez."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_metricas_test_externo_60d.csv"
        ),
        "uso_en_memoria": (
            "Resultados finales de generalización"
        )
    },
    {
        "bloque": "Matriz de confusión",
        "afirmacion_verificable": (
            "Se obtuvieron 9.933 TN, 90 FP, "
            "20 FN y 2.958 TP."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_predicciones_test_externo_60d.csv"
        ),
        "uso_en_memoria": (
            "Interpretación de aciertos y errores"
        )
    },
    {
        "bloque": "Interpretabilidad",
        "afirmacion_verificable": (
            "Se calcularon los 176 coeficientes "
            "estandarizados del modelo."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_coeficientes_logistic_l2_60d.csv"
        ),
        "uso_en_memoria": (
            "Explicación de las variables relevantes"
        )
    },
    {
        "bloque": "Estabilidad",
        "afirmacion_verificable": (
            "Las veinte variables principales conservaron "
            "el signo en los cinco pliegues."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_estabilidad_coeficientes_60d.csv"
        ),
        "uso_en_memoria": (
            "Robustez de la interpretación"
        )
    },
    {
        "bloque": "Calibración",
        "afirmacion_verificable": (
            "El modelo obtuvo un Brier skill score "
            "de aproximadamente 0,9714."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_resumen_calibracion_riesgo_60d.json"
        ),
        "uso_en_memoria": (
            "Calidad de las probabilidades"
        )
    },
    {
        "bloque": "Utilidad operativa",
        "afirmacion_verificable": (
            "El 25 % de clientes con mayor riesgo "
            "concentra el 99,63 % del churn."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "modelado_escenarios_operativos_60d.csv"
        ),
        "uso_en_memoria": (
            "Escenarios de priorización"
        )
    },
    {
        "bloque": "Modelo reproducible",
        "afirmacion_verificable": (
            "El pipeline final, las variables y el umbral "
            "se almacenaron conjuntamente."
        ),
        "notebook_origen": (
            "04_modelado.ipynb"
        ),
        "archivo_evidencia": (
            "modelos/"
            "modelo_final_logistic_l2_60d.joblib"
        ),
        "uso_en_memoria": (
            "Reproducibilidad y posible despliegue"
        )
    },
    {
        "bloque": "Trazabilidad metodológica",
        "afirmacion_verificable": (
            "Las decisiones y sus criterios quedaron "
            "documentados explícitamente."
        ),
        "notebook_origen": (
            "05_sintesis_final.ipynb"
        ),
        "archivo_evidencia": (
            "resultados/"
            "tabla_registro_decisiones_metodologicas.csv"
        ),
        "uso_en_memoria": (
            "Defensa metodológica ante el tribunal"
        )
    }
])


# ---------------------------------------------------------
# Comprobación de los archivos del mapa
# ---------------------------------------------------------

mapa_evidencias[
    "ruta_absoluta"
] = (
    mapa_evidencias[
        "archivo_evidencia"
    ]
    .map(
        lambda ruta:
        RUTA_PROYECTO / Path(ruta)
    )
)

mapa_evidencias[
    "existe"
] = (
    mapa_evidencias[
        "ruta_absoluta"
    ]
    .map(
        lambda ruta: ruta.exists()
    )
)

mapa_evidencias[
    "tamano_mb"
] = (
    mapa_evidencias[
        "ruta_absoluta"
    ]
    .map(
        lambda ruta:
        (
            ruta.stat().st_size
            / (1024 ** 2)
        )
        if ruta.exists()
        else np.nan
    )
)

if not mapa_evidencias[
    "existe"
].all():
    archivos_faltantes = (
        mapa_evidencias.loc[
            ~mapa_evidencias["existe"],
            "archivo_evidencia"
        ]
        .tolist()
    )

    raise FileNotFoundError(
        "Faltan evidencias del mapa: "
        f"{archivos_faltantes}"
    )


# ---------------------------------------------------------
# Función para calcular la huella SHA-256
# ---------------------------------------------------------

def calcular_sha256(ruta_archivo):
    """
    Calcula la huella SHA-256 leyendo el archivo
    por bloques para evitar cargarlo completo.
    """
    generador_hash = hashlib.sha256()

    with open(
        ruta_archivo,
        "rb"
    ) as archivo:

        while True:
            bloque = archivo.read(
                1024 * 1024
            )

            if not bloque:
                break

            generador_hash.update(
                bloque
            )

    return generador_hash.hexdigest()


mapa_evidencias[
    "sha256"
] = (
    mapa_evidencias[
        "ruta_absoluta"
    ]
    .map(
        calcular_sha256
    )
)

mapa_evidencias[
    "ruta_absoluta"
] = (
    mapa_evidencias[
        "ruta_absoluta"
    ]
    .astype(str)
)


# ---------------------------------------------------------
# Presentación del mapa de trazabilidad
# ---------------------------------------------------------

print(
    "Evidencias registradas:",
    len(mapa_evidencias)
)

display(
    mapa_evidencias[
        [
            "bloque",
            "afirmacion_verificable",
            "notebook_origen",
            "archivo_evidencia",
            "existe",
            "tamano_mb"
        ]
    ]
    .style.format({
        "tamano_mb": "{:.3f}"
    })
)


# ---------------------------------------------------------
# Archivos de cierre
# ---------------------------------------------------------

RUTA_MAPA_EVIDENCIAS = (
    RUTA_RESULTADOS
    / "mapa_trazabilidad_evidencias.csv"
)

RUTA_ACTA_CIERRE = (
    RUTA_RESULTADOS
    / "acta_cierre_tecnico.json"
)

RUTA_README_RESULTADOS = (
    RUTA_RESULTADOS
    / "README_RESULTADOS_TFM.md"
)

mapa_evidencias.to_csv(
    RUTA_MAPA_EVIDENCIAS,
    index=False
)


# ---------------------------------------------------------
# Acta técnica de cierre
# ---------------------------------------------------------

acta_cierre_tecnico = {
    "status": "technical_project_closed",
    "generated_at": (
        datetime.now()
        .astimezone()
        .isoformat()
    ),
    "project_root": str(
        RUTA_PROYECTO
    ),
    "notebooks": (
        comprobacion_notebooks
        .to_dict(
            orient="records"
        )
    ),
    "final_configuration": {
        "window_days": 60,
        "model": "logistic_l2",
        "threshold": 0.20,
        "features": 176
    },
    "external_test": {
        "clients": 13_001,
        "average_precision": float(
            metricas_finales[
                "average_precision"
            ]
        ),
        "auc_roc": float(
            metricas_finales[
                "auc_roc"
            ]
        ),
        "precision": float(
            metricas_finales[
                "precision"
            ]
        ),
        "recall": float(
            metricas_finales[
                "recall"
            ]
        ),
        "f1": float(
            metricas_finales["f1"]
        ),
        "f2": float(
            metricas_finales["f2"]
        )
    },
    "traceability_records": int(
        len(mapa_evidencias)
    ),
    "all_evidence_files_exist": bool(
        mapa_evidencias[
            "existe"
        ].all()
    ),
    "configuration_changed_after_test": False,
    "further_model_tuning_allowed": False,
    "main_limitation": (
        "La construcción temporal original de la "
        "etiqueta churn no está completamente documentada."
    )
}

with open(
    RUTA_ACTA_CIERRE,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        acta_cierre_tecnico,
        archivo,
        ensure_ascii=False,
        indent=4
    )


# ---------------------------------------------------------
# README de resultados
# ---------------------------------------------------------

texto_readme_resultados = f"""# Resultados finales del TFM

## Configuración definitiva

- Ventana de observación: 60 días.
- Variables predictoras: 176.
- Modelo: regresión logística con regularización L2.
- Umbral: 0,20.
- Clientes de entrenamiento final: 52.004.
- Clientes de test externo: 13.001.

## Métricas del test externo

- Average precision: {metricas_finales['average_precision']:.4f}.
- AUC-ROC: {metricas_finales['auc_roc']:.4f}.
- Precision: {metricas_finales['precision']:.4f}.
- Recall: {metricas_finales['recall']:.4f}.
- F1: {metricas_finales['f1']:.4f}.
- F2: {metricas_finales['f2']:.4f}.

## Archivos principales

- `modelado_configuracion_final_60d.json`: configuración congelada.
- `modelado_metricas_test_externo_60d.csv`: métricas definitivas.
- `modelado_coeficientes_logistic_l2_60d.csv`: interpretabilidad.
- `modelado_escenarios_operativos_60d.csv`: priorización de clientes.
- `mapa_trazabilidad_evidencias.csv`: relación entre conclusiones y evidencias.
- `acta_cierre_tecnico.json`: estado final del proyecto.
- `../modelos/modelo_final_logistic_l2_60d.joblib`: pipeline entrenado.

## Regla de uso

El test externo ya fue evaluado y no se permiten nuevos ajustes del modelo,
del umbral o de las variables basados en sus resultados.

## Limitación principal

La construcción temporal original de la etiqueta `churn` no está
completamente documentada. Los resultados demuestran generalización sobre
la partición externa proporcionada, pero no prueban una anticipación
temporal estricta en producción.
"""

with open(
    RUTA_README_RESULTADOS,
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(
        texto_readme_resultados
    )


# ---------------------------------------------------------
# Validación definitiva del proyecto técnico
# ---------------------------------------------------------

cierre_tecnico_valido = (
    sintesis_final_valida
    and registro_metodologico_valido
    and len(comprobacion_notebooks) == 5
    and comprobacion_notebooks[
        "existe"
    ].all()
    and len(mapa_evidencias) == 14
    and mapa_evidencias[
        "existe"
    ].all()
    and mapa_evidencias[
        "sha256"
    ].str.len().eq(64).all()
    and RUTA_MAPA_EVIDENCIAS.exists()
    and RUTA_ACTA_CIERRE.exists()
    and RUTA_README_RESULTADOS.exists()
    and acta_cierre_tecnico[
        "status"
    ] == "technical_project_closed"
    and acta_cierre_tecnico[
        "configuration_changed_after_test"
    ] is False
    and acta_cierre_tecnico[
        "further_model_tuning_allowed"
    ] is False
)


# ---------------------------------------------------------
# Presentación final
# ---------------------------------------------------------

print(
    "\nNotebooks comprobados:",
    len(comprobacion_notebooks)
)

print(
    "Evidencias trazadas:",
    len(mapa_evidencias)
)

print(
    "Configuración final:",
    "60 días + logistic_l2 + umbral 0.20"
)

print(
    "Estado del proyecto:",
    acta_cierre_tecnico[
        "status"
    ]
)

print(
    "¿Se permiten nuevos ajustes del modelo?",
    acta_cierre_tecnico[
        "further_model_tuning_allowed"
    ]
)

print(
    "¿El proyecto técnico se ha cerrado "
    "correctamente?",
    cierre_tecnico_valido
)